In [1]:
import matplotlib.pyplot as plt
import io
import base64
from IPython.display import display, HTML

# Добавляем CSS для глобального применения прокрутки к графикам
display(HTML("""
    <style>
        .scrollable-output {
            max-height: 300px;
            overflow-y: auto;
            border: 1px solid #ddd;
            padding: 5px;
        }
    </style>
"""))

def scrollable_plot(fig, height=400):
    """Выводит matplotlib figure в прокручиваемом контейнере"""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches='tight')
    buf.seek(0)
    img_base64 = base64.b64encode(buf.getvalue()).decode()
    
    html = f"""
    <div class="scrollable-output" style="max-height: {height}px;">
        <img src="data:image/png;base64,{img_base64}">
    </div>
    """
    plt.close(fig)  # Закрываем фигуру, чтобы избежать двойного вывода
    display(HTML(html))

# Пример использования
# fig, ax = plt.subplots(figsize=(10, 5))
# ax.plot(range(10), range(10))
# scrollable_plot(fig)


In [2]:
import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
%matplotlib inline
plt.rcParams["image.cmap"] = 'magma'
import seaborn as sns
from tqdm.notebook import tqdm
from tabulate import tabulate

from sklearn.metrics.pairwise import cosine_similarity

from damo.config.base import parse_config
from damo.detectors.detector import build_local_model
from tools.demo import Infer

from my_help_functions.hooks import register_hooks_batch_forward
from my_help_functions.cosine_matrix import get_positions_of_classes_on_flattened_image, get_positions_of_classes_on_flattened_image_for_collage


In [3]:
def load_images_from_folder(folder):
    img_list = []
    names = []
    for filename in sorted(os.listdir(folder)):
        if filename.lower().endswith(".jpg"):
            path = os.path.join(folder, filename)
            img = cv2.imread(path)
            if img is not None:
                img_list.append(img)
                names.append(filename)
    return img_list, names

In [4]:
from damo.dataset.transforms import transforms as T
from damo.structures.image_list import to_image_list

def transform_img_(origin_img, size_divisibility=0, image_max_range=(640, 640), flip_prob=0.0,
                  image_mean=[0.0, 0.0, 0.0], image_std=[1.0, 1.0, 1.0], keep_ratio=False, infer_size=[640,640]):
    transform = [
        T.Resize(image_max_range, target_size=infer_size, keep_ratio=keep_ratio),
        T.RandomHorizontalFlip(flip_prob),
        T.ToTensor(),
        T.Normalize(mean=image_mean, std=image_std),
    ]
    transform = T.Compose(transform)

    img, _ = transform(origin_img)
    img = to_image_list(img, size_divisibility)
    return img

In [5]:
def run_batch_inference(config_path, ckpt_path, img_folder, model_desired_layers, device='cuda'):
    config = parse_config(config_path)
    model = build_local_model(config, device)
    model.eval()
    model.to(device)
    layer_outputs_fwd = register_hooks_batch_forward(model, model_desired_layers)

    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model'])

    imgs, names = load_images_from_folder(img_folder)
    transformed = [transform_img_(img) for img in imgs]

    image_list = torch.cat([img.tensors for img in transformed], dim=0).to(device)

    with torch.no_grad():
        outputs = model(image_list)

    bboxes = []
    scores = []
    cls_inds = []
    for i in range(len(outputs)):
        bboxes.append(outputs[i].bbox)
        scores.append(outputs[i].get_field('scores'))
        cls_inds.append(outputs[i].get_field('labels'))

    return bboxes, scores, cls_inds, layer_outputs_fwd, model


In [6]:
config_path = './configs/damoyolo_tinynasL20_T.py'
ckpt_path = './weights/damoyolo_tiny.pth'
img_folder = './downloaded_images'

layers_to_add = ['backbone.block_list.3.block_list.0.conv1.conv1',
                 'backbone.block_list.3.block_list.0.conv1.bn1'  
                ]

bboxes, scores, cls_inds, fms, model = run_batch_inference(config_path, ckpt_path, img_folder, layers_to_add)

forward hook used
forward hook used


In [7]:
bn = model.backbone.block_list[3].block_list[0].conv1.bn1
convolution = model.backbone.block_list[3].block_list[0].conv1.conv1

In [8]:
from tabulate import tabulate

data = [(layer, str(embeddings[0].shape), str(embeddings[1].shape)) for layer, embeddings in fms.items()]
print(tabulate(data, headers=["Слой", "Размер входа", "Размер выхода"], tablefmt="grid"))

+----------------------------------------------------------------------------------+------------------------------+------------------------------+
| Слой                                                                             | Размер входа                 | Размер выхода                |
+==================================================================================+==============================+==============================+
| Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)                    | torch.Size([80, 96, 80, 80]) | torch.Size([80, 96, 80, 80]) |
+----------------------------------------------------------------------------------+------------------------------+------------------------------+
| BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True) | torch.Size([80, 96, 80, 80]) | torch.Size([80, 96, 80, 80]) |
+----------------------------------------------------------------------------------+------------------------------+---

In [9]:
conv = {}
conv['before conv'] = list(fms.values())[0][0]
conv['after conv']  = list(fms.values())[0][1]
conv['after center'] = list(fms.values())[1][0] - bn.running_mean.view(-1, 1, 1)

center = bn.running_mean.view(-1, 1, 1)


In [10]:
relu = torch.nn.ReLU(inplace=False)

after_conv = conv['after conv']

after = (after_conv - bn.running_mean.view(-1, 1, 1)) / torch.sqrt(bn.running_var.view(-1, 1, 1) + bn.eps)
after = (torch.sign(bn.weight.view(-1, 1, 1)) * after) + (bn.bias.view(-1, 1, 1) / torch.abs(bn.weight.view(-1, 1, 1)))
conv['after relu'] = relu(after)

In [11]:
for i, j in conv.items():
    print(i, j.shape)

before conv torch.Size([80, 96, 80, 80])
after conv torch.Size([80, 96, 80, 80])
after center torch.Size([80, 96, 80, 80])
after relu torch.Size([80, 96, 80, 80])


In [12]:
import torch.nn.functional as F

def pairwise_cosine_similarity(x1, x2):
    x1 = F.normalize(x1, dim=0)
    x2 = F.normalize(x2, dim=0)
    return x1.T @ x2

In [13]:
batch_center_before_conv  = conv['before conv'].mean(dim=(0, 2, 3))
batch_center_through_conv = convolution(batch_center_before_conv.view(1, -1, 1, 1)).squeeze()
batch_center_after_conv   = conv['after conv'].mean(dim=(0, 2, 3)).squeeze()

forward hook used


In [14]:
W = convolution.weight.data
W_eff = W.view(96, 96, -1).sum(dim=2)  # shape: [out_ch, in_ch]
W_pinv = torch.pinverse(W_eff)
center_before_conv = (W_pinv @ (center.squeeze()))

center = center.squeeze()

In [15]:
W_pinv.size()
W = W.view(96,96)
torch.linalg.norm(W @ W_pinv)
torch.linalg.norm(torch.eye(96))

tensor(9.7980)

In [16]:
print(pairwise_cosine_similarity(batch_center_before_conv, center_before_conv))
print(pairwise_cosine_similarity(batch_center_through_conv, center))
print(pairwise_cosine_similarity(batch_center_through_conv, batch_center_after_conv))

tensor(0.9484, device='cuda:0')
tensor(0.8914, device='cuda:0', grad_fn=<DotBackward>)
tensor(1., device='cuda:0', grad_fn=<DotBackward>)


In [17]:
print(torch.linalg.norm(batch_center_before_conv  - center_before_conv))
print(torch.linalg.norm(batch_center_through_conv - center))
print(torch.linalg.norm(batch_center_through_conv - batch_center_after_conv))

tensor(2.5291, device='cuda:0')
tensor(0.6648, device='cuda:0', grad_fn=<CopyBackwards>)
tensor(6.4047e-07, device='cuda:0', grad_fn=<CopyBackwards>)


In [18]:
print(torch.linalg.norm(center_before_conv))
print(torch.linalg.norm(batch_center_before_conv))
print(torch.linalg.norm(batch_center_through_conv))
print(torch.linalg.norm(center))
print(torch.linalg.norm(batch_center_after_conv))

tensor(7.7698, device='cuda:0')
tensor(7.9389, device='cuda:0')
tensor(1.4586, device='cuda:0', grad_fn=<CopyBackwards>)
tensor(1.3717, device='cuda:0')
tensor(1.4586, device='cuda:0')
